# Generation of Background Events

## Load Libraries

In [1]:
import pandas as pd
import os
import pythia8
import numpy as np
import json

## Function that Generates Events for fixed Energy 

In [2]:
def generate_events(energy, nevent=100000, level='hadron', target='W', process='CC_numu', felix_user=False):
        
    # Setup Pythia
    pythia = pythia8.Pythia()

    # Set random seeds
    pythia.readString("Random:setSeed=on")
    pythia.readString("Random:seed=0")

    # Define beams (frameType = 2 for fixed target collisions)
    pythia.readString("Beams:frameType = 2")
    pythia.readString("Beams:eA = "+str(energy))
    pythia.readString("Beams:eB = 0")

    # Use the nuclear PDF nCTEQ15 for Tungsten (A=184, Z=74)
    if target == 'W': pdffile = "nCTEQ15FullNuc_184_74_0000.dat"
    elif target == 'Fe': pdffile = "nCTEQ15FullNuc_56_26_0000.dat"    
    if felix_user: 
        pdfpath = "/Users/felixkling/work/LHAPDF-6.2.1/installed/share/LHAPDF/"+pdffile[:-9]+"/"
        pythia.readString("PDF:pSet=LHAGrid1:"+pdfpath+pdffile);
    else:
        pythia.readString("PDF:pSet=lhagrid1:"+pdffile)

    # Activate necessary options for Pythia
    pythia.readString("PDF:lepton = off")
    pythia.readString("TimeShower:QEDshowerByL = off")

    # Fix Renormalization/Factorization scale
    pythia.readString("SigmaProcess:factorScale2=6")
    pythia.readString("SigmaProcess:renormScale2=6")

    # Minimal phase space cuts
    pythia.readString("PhaseSpace:mHatMin=1")
    pythia.readString("PhaseSpace:pTHatMin=0")
    pythia.readString("PhaseSpace:pTHatMinDiverge=0")

    # Define process type and set the beam accordingly
    if process == "CC_numu":
        pythia.readString("WeakBosonExchange:ff2ff(t:W)=on")
        pythia.readString("Beams:idA = 14")
    elif process == "CC_nuebar":
        pythia.readString("WeakBosonExchange:ff2ff(t:W)=on")
        pythia.readString("Beams:idA = -12")  
    elif process == "NC_numu":
        pythia.readString("WeakBosonExchange:ff2ff(t:gmZ)=on")
        pythia.readString("Beams:idA = 14")
    elif process == "NC_nuebar":
        pythia.readString("WeakBosonExchange:ff2ff(t:gmZ)=on")
        pythia.readString("Beams:idA = -12")  
    else:
        print("Error: process_type = " + process + " is not valid.")
        return pd.DataFrame()
    
    # Decide if shower and hadronization should be included
    if level == "hard":
        pythia.readString("HadronLevel:all=off")
        pythia.readString("PartonLevel:all=off")
    elif level == "parton":
        pythia.readString("HadronLevel:all=off")
        pythia.readString("PartonLevel:all=on")
    elif level == "hadron":
        pythia.readString("HadronLevel:all=on")
        pythia.readString("PartonLevel:all=on")
    else:
        print("Error: level = "+level+" is not a valid option")
        return False

    # Initialize
    pythia.init()

    # List to store events data for DataFrame
    events_data = []

    # Loop over events
    for ievent in range(nevent):
        
        # Generate next event
        if not pythia.next(): continue
        particles = pythia.process if level == "hard" else pythia.event

        # Loop through particles in event
        found_muplus, event_data = False, [] 
        iiparticle = 0
        for iparticle, particle in enumerate(particles):
            
            # Reject non-final state particles
            if particle.status() < 0: continue
            # Collect particle data
            pid = particle.id()
            px, py = round(particle.px(),3), round(particle.py(),3)
            pz, e = round(particle.pz(),3), round(particle.e(),3)
            parent_pid1 = particles[particle.mother1()].id()
            parent_pid2 = particles[particle.mother2()].id()

            # Append particle data as a new row
            event_data.append([ievent, iiparticle, energy, pid, px, py, pz, e, parent_pid1, parent_pid2])
            iiparticle+=1
            if pid == -13: found_muplus = True

        # if mu+ found, save 
        if found_muplus: 
            for x in event_data: events_data.append(x)
                
    # Create output directory corresponding to different process
    output_dir = "output_events_"+process
    if not os.path.isdir(output_dir): os.makedirs(output_dir)
    
    # Convert collected event data to DataFrame
    columns = ["ievent", "iparticle", "truth_energy", "pid", "px", "py", "pz", "e", "parent_pid1", "parent_pid2"]
    df = pd.DataFrame(events_data, columns=columns)
    
    # See if data frame already exist, if yes, merge. 
    csv_file = output_dir+"/events_"+str(energy)+".csv.zip"
    if os.path.exists(csv_file): 
        df_old = pd.read_csv(csv_file, compression="zip")
        nevt_old = df_old['ievent'].max()+1
        df['ievent'] = df['ievent']+ nevt_old
        df = pd.concat([df_old, df]) 
        
    # Export
    df.to_csv(csv_file, index=False, compression='zip')
    print("Saved events for energy "+str(energy)+" to "+csv_file)
    
    # Save number of generated events
    logging_filename = output_dir+"/generated_events.json"
    if os.path.exists(logging_filename): 
        with open(logging_filename, "r") as f: 
            logging_file = json.load(f)
    else: logging_file = {}
    if str(energy) in logging_file: logging_file[str(energy)] += nevent
    else: logging_file[str(energy)] = nevent
    with open(logging_filename, "w") as f: 
        json.dump(logging_file, f, indent=4)

## Use Function to Generate 100 Events per Energy

In [4]:
# List of energies
energies = [
    14.21, 17.90, 22.53, 28.37, 35.71, 44.964, 56.60,
    71.26, 89.71, 112.94, 142.19, 179.00, 225.35,
    283.70, 357.16, 449.64, 566.07, 712.64, 897.16,
    1129.46, 1421.90, #1790.077754, 2253.574373, 2837.082046, 3571.674683,
    #4496.472021, 5660.722891, 7126.427896, 8971.641174
] 
processes = ["CC_numu", "CC_nuebar", "NC_numu", "NC_nuebar"]

# Loop through all energies
for process in processes: 
    for energy in energies:
        generate_events(energy=energy, nevent=100000, level='hadron', target='Fe', process=process, felix_user=True)

Saved events for energy 14.21 to output_events_CC_numu/events_14.21.csv.zip
Saved events for energy 17.9 to output_events_CC_numu/events_17.9.csv.zip
Saved events for energy 22.53 to output_events_CC_numu/events_22.53.csv.zip
Saved events for energy 28.37 to output_events_CC_numu/events_28.37.csv.zip
Saved events for energy 35.71 to output_events_CC_numu/events_35.71.csv.zip
Saved events for energy 44.964 to output_events_CC_numu/events_44.964.csv.zip
Saved events for energy 56.6 to output_events_CC_numu/events_56.6.csv.zip
Saved events for energy 71.26 to output_events_CC_numu/events_71.26.csv.zip
Saved events for energy 89.71 to output_events_CC_numu/events_89.71.csv.zip
Saved events for energy 112.94 to output_events_CC_numu/events_112.94.csv.zip
Saved events for energy 142.19 to output_events_CC_numu/events_142.19.csv.zip
Saved events for energy 179.0 to output_events_CC_numu/events_179.0.csv.zip
Saved events for energy 225.35 to output_events_CC_numu/events_225.35.csv.zip
Saved ev

## Generate 100 more Events for E=1421.909302

In [ ]:
generate_events(energy=1421.90, nevent=1000, level='hadron', target='Fe', process='CC_numu', felix_user=True)